In [1]:
import pandas as pd
import requests

In [2]:
url = 'https://statusinvest.com.br/category/advancedsearchresultpaginated?search=%7B%22Sector%22%3A%22%22%2C%22SubSector%22%3A%22%22%2C%22Segment%22%3A%22%22%2C%22my_range%22%3A%22-20%3B100%22%2C%22forecast%22%3A%7B%22upsidedownside%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22estimatesnumber%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22revisedup%22%3Atrue%2C%22reviseddown%22%3Atrue%2C%22consensus%22%3A%5B%5D%7D%2C%22dy%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_l%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22peg_ratio%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_vp%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_ativo%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22margembruta%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22margemebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22margemliquida%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_ebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22ev_ebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22dividaliquidaebit%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22dividaliquidapatrimonioliquido%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_sr%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_capitalgiro%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22p_ativocirculante%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22roe%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22roic%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22roa%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22liquidezcorrente%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22pl_ativo%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22passivo_ativo%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22giroativos%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22receitas_cagr5%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22lucros_cagr5%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22liquidezmediadiaria%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22vpa%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22lpa%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%2C%22valormercado%22%3A%7B%22Item1%22%3Anull%2C%22Item2%22%3Anull%7D%7D&orderColumn=&isAsc=&page=0&take=617&CategoryType=1'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
data = response.json()
indicadores = pd.DataFrame(data["list"])

In [3]:
screening = indicadores[['companyname', 'ticker', 'price','dy', 'p_l', 'margemliquida', 'dividaliquidaebit','roe', 'roic','liquidezmediadiaria']]

In [4]:
#Tirando as empresas com liquidez diária menor que 1.000.000
screening = screening[screening['liquidezmediadiaria'] >=1_000_000]

In [5]:
# Filtro de Joel Greenblatt pelo ROIC
screening = screening[screening['roic'] >=8 ]

In [6]:
# Filtro de Dividend Yield pelo Décio Bazin
screening = screening[screening['dy'] >=5]

In [7]:
# Filtro para não pagar caro na ação
screening = screening[screening['p_l'] <=15]

In [8]:
# usar bom senso qualitativo
screening = screening[screening['margemliquida'] >=5]

In [9]:
screening = screening[screening['dividaliquidaebit'] <=5]

In [10]:
screening = screening[screening['roe'] >=9.5]

In [ ]:
# Ranquear Dividend Yield do maior para o menor
screening = screening.sort_values("dy", ascending=False)
screening["ranking_dy"] = range(1, len(screening) + 1)

In [ ]:
# Ranquear P/L do menor para o maior
screening = screening.sort_values("p_l")
screening["ranking_p/l"] = range(1, len(screening) + 1)

In [ ]:
# Ranquear Margem Líquida do maior para o menor
screening = screening.sort_values("margemliquida", ascending=False)
screening["ranking_marg_liq"] = range(1, len(screening) + 1)

In [ ]:
# Ranquear Dívida liquida sobre EBIT do menor para o maior
screening = screening.sort_values("dividaliquidaebit")
screening["ranking_divida_liquida/ebit"] = range(1, len(screening) + 1)

In [18]:
# Ranquear ROE do maior para o menor
screening = screening.sort_values("roe", ascending=False)
screening["ranking_roe"] = range(1, len(screening) + 1)

In [22]:
screening["score_final"] = screening[["ranking_dy", "ranking_p/l", "ranking_marg_liq", "ranking_divida_liquida/ebit",'ranking_roe' ]].sum(axis=1)

In [24]:
screening = screening.sort_values("score_final")

Com essses filtros, é possível escolher quais ações analisar os relatórios de forma performática!

Depois disso, usar o preço teto projetivo do Bazin nessas ações para saber o quão caro está

In [25]:
screening

,companyname,ticker,price,dy,p_l,margemliquida,dividaliquidaebit,roe,roic,liquidezmediadiaria,ranking_dy,ranking_p/l,ranking_marg_liq,ranking_divida_liquida/ebit,ranking_roe,score_final
604,VULCABRAS,VULC3,15.62,31.7379,4.2595,32.73,1.03,48.01,12.88,1.297049e+07,2,3,6,26,3,40
299,GRENDENE,GRND3,4.37,35.1551,6.1281,24.95,-3.04,20.44,8.04,1.491550e+07,1,9,9,2,21,42
494,RIACHUELO,RIAA3,9.74,24.3492,3.3064,14.05,0.06,27.56,25.71,1.470169e+07,3,2,23,12,10,50
352,LAVVI EMPREENDIMENTOS IMOBILIÁRIOS S.A.,LAVV3,13.32,20.2633,6.2396,23.51,0.85,23.25,13.52,1.042048e+07,4,11,11,24,14,64
203,CURY CONSTRUTORA E INCORPORADORA S.A.,CURY3,29.75,14.2057,9.3658,18.07,-0.25,58.69,35.99,1.090871e+08,13,27,15,10,2,67
212,DIRECIONAL ENGENHARIA,DIRR3,12.98,17.0907,8.5525,18.18,0.66,34.02,17.50,8.235130e+07,7,23,14,20,5,69
375,MOURA DUBEUX ENGENHARIA S/A,MDNE3,30.50,17.8272,7.4434,17.84,0.73,27.72,16.12,3.798368e+07,6,18,16,21,9,70
12,ALLIED TECNOLOGIA S.A.,ALLD3,6.48,18.1295,1.8430,6.04,0.08,21.44,17.95,3.198158e+06,5,1,42,14,17,79
522,SCHULZ,SHUL4,5.09,12.2900,6.3306,14.91,-0.56,19.27,9.24,1.936810e+06,16,13,21,6,23,79
460,MARCOPOLO,POMO3,6.12,16.8576,6.2426,13.51,1.09,31.40,12.17,3.620351e+06,9,12,25,28,6,80
